In [1]:
!git clone https://github.com/facebookresearch/sapiens.git

fatal: destination path 'sapiens' already exists and is not an empty directory.


In [ ]:
# Quiet setup: the full output goes to setup.log, only one line per step is printed
!conda env list | grep -q "^sapiens " || conda create -n sapiens python=3.10 -y -q > setup.log 2>&1 && echo "env: ok" || echo "env: FAILED (see setup.log)"
!conda run -n sapiens python -m pip install -q ipykernel >> setup.log 2>&1 && echo "ipykernel: ok" || echo "ipykernel: FAILED (see setup.log)"
!conda run -n sapiens python -m ipykernel install --user --name sapiens --display-name "Python (Sapiens)" >> setup.log 2>&1 && echo "kernel: ok" || echo "kernel: FAILED (see setup.log)"
!conda run -n sapiens python -m pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128 >> setup.log 2>&1 && echo "torch: ok" || echo "torch: FAILED (see setup.log)"
!conda run -n sapiens python -m pip install -q numpy opencv-python pandas matplotlib huggingface_hub >> setup.log 2>&1 && echo "packages: ok" || echo "packages: FAILED (see setup.log)"

In [4]:
from huggingface_hub import login
from huggingface_hub import model_info

login()

info = model_info("noahcao/sapiens-pose-coco")

print("Authenticated successfully!")
print("Model:", info.id)

Authenticated successfully!
Model: noahcao/sapiens-pose-coco


In [13]:
from pathlib import Path

SAPIENS_ROOT = Path("sapiens").resolve()
LITE_ROOT = SAPIENS_ROOT / "lite"
SCRIPT_DIR = LITE_ROOT / "scripts" / "demo" / "torchscript"

script = SCRIPT_DIR / "pose_keypoints133.sh"

[FOUND] /home/ausjak2/saugios-rankos/sapiens/lite/demo/vis_pose.py
[MISSING] /home/ausjak2/saugios-rankos/sapiens/lite/pose/demo/mmdetection_cfg/rtmdet_m_640-8xb32_coco-person_no_nms.py
[FOUND] /home/ausjak2/saugios-rankos/sapiens/lite/scripts/demo/torchscript/pose_keypoints133.sh
[MISSING] /home/ausjak2/sapiens_lite_host/torchscript/pose/checkpoints/sapiens_1b/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2
[MISSING] /home/ausjak2/sapiens_lite_host/torchscript/detector/checkpoints/rtmpose/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth


In [14]:
from huggingface_hub import hf_hub_download
from pathlib import Path

CHECKPOINT_ROOT = (
    Path.home() / "sapiens_lite_host" / "torchscript"
)

MODEL_DIR = (
    CHECKPOINT_ROOT / "pose" / "checkpoints" / "sapiens_1b"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

FILENAME = (
    "sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2"
)

MODEL_PATH = hf_hub_download(
    repo_id="noahcao/sapiens-pose-coco",
    filename=(
        "sapiens_lite_host/torchscript/pose/checkpoints/"
        f"sapiens_1b/{FILENAME}"
    ),
    local_dir=str(Path.home() / "sapiens_download")
)

Downloaded: /home/ausjak2/sapiens_download/sapiens_lite_host/torchscript/pose/checkpoints/sapiens_1b/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2


In [15]:
from pathlib import Path
import shutil

SOURCE = Path(
    "/home/ausjak2/sapiens_download/sapiens_lite_host/torchscript/"
    "pose/checkpoints/sapiens_1b/"
    "sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2"
)

DEST_DIR = Path.home() / "sapiens_lite_host/torchscript/pose/checkpoints/sapiens_1b"
DEST_DIR.mkdir(parents=True, exist_ok=True)

DEST = DEST_DIR / SOURCE.name

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

shutil.copy2(SOURCE, DEST)


Checkpoint: /home/ausjak2/sapiens_lite_host/torchscript/pose/checkpoints/sapiens_1b/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2
Size: 4.36 GB


In [7]:
DATASET_ROOT = Path("dataset")

VIDEO_DIR = DATASET_ROOT / "videos" / "needed"
VIDEO_INFO_DIR = DATASET_ROOT / "video_info" / "video_info"

RESULTS_DIR = Path("sapiens_results")
FRAMES_DIR = RESULTS_DIR / "frames"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"

for folder in [FRAMES_DIR, PREDICTIONS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [8]:
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}

video_files = sorted([
    f for f in VIDEO_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in VIDEO_EXTENSIONS
])

In [9]:
info_files = [
    f for f in VIDEO_INFO_DIR.rglob("*")
    if f.is_file()
]

print(f"Found {len(info_files)} information files")

for f in info_files[:30]:
    print(f.name, f.suffix)

Found 0 information files


In [ ]:
%pip install -q json_tricks tqdm
import sys

In [ ]:
import pandas as pd

SIGN = "BORED1"    # which sign to look at

# only videos that are really in the dataset folder
info = pd.read_csv(VIDEO_INFO_DIR / "all_videos.txt")
info = info[info["Video file"].isin([v.name for v in video_files])]
print("signs you can use:", sorted(info["Gloss"].unique()))

# the same sign done by 3 different people
info = info[info["Gloss"] == SIGN].drop_duplicates("Participant ID").head(3)
videos = [v for v in video_files if v.name in set(info["Video file"])]
print(info[["Participant ID", "Video file"]])
assert videos, f"No videos of {SIGN} - pick one of the signs listed above"

for video in videos:
    !mkdir -p sapiens_results/frames/{video.stem}
    !ffmpeg -loglevel error -i "{video}" sapiens_results/frames/{video.stem}/%05d.jpg
    !cd sapiens/lite && {sys.executable} demo/vis_pose.py {DEST} --num_keypoints 133 --radius 3 --input ../../sapiens_results/frames/{video.stem} --output-root ../../sapiens_results/output/{video.stem}

In [ ]:
from IPython.display import Image, display

# put Sapiens' output frames together into an animated GIF (GIFs show in every notebook viewer)
for video in videos:
    frames = sorted(Path(f"sapiens_results/output/{video.stem}").glob("*.jpg"))
    print(video.name, "-", len(frames), "frames from Sapiens")
    if not frames:
        continue
    out = f"sapiens_results/{video.stem}_sapiens.gif"
    !ffmpeg -loglevel error -y -framerate 30 -i sapiens_results/output/{video.stem}/%05d.jpg -vf "fps=15,scale=400:-1" {out}
    display(Image(filename=out))
